# Passive board monitoring

Run setup and connection, then **YJ**, **HK**, or both dashboard cells. Leave this
notebook running while working in another notebook. Run All starts both displays.

This notebook listens to existing MQTT broadcasts. It does not start or stop a
measurement, power a detector or laser, change routing, or change settings.
The optional status cell is the only cell that sends a board command.

Calibration currently stops ordinary throughput streaming: its progress still
appears in the message pane, while plots show the age of the last sample. Waiting
and stale displays do not imply zero light. Only received boot events are shown.
Plots retain 600 samples per channel; the shared collector retains at most 40,000
samples and the message pane retains 500 messages. Nothing is saved automatically.

Use the workspace `.venv` kernel in browser Jupyter. Run the setup cell separately
on a fresh kernel before connecting and displaying the dashboards.


In [ ]:
%matplotlib widget
from pathlib import Path
from collections import deque
from contextlib import suppress
from html import escape
import asyncio
import logging
import sys

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

TOOLS = next((p.resolve()
              for root in (Path.cwd(), *Path.cwd().parents)
              for p in (root, root / 'tools', root / 'hispec-tib/tools')
              if (p / 'hispec_fibpcb.py').is_file()), None)
if TOOLS is None:
    raise RuntimeError('Open this notebook from the repository or its tools directory.')
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
import hispec_fibpcb as hspcb

BROKER, DEVICE = 'hispec.caltech.edu', 'hsfib-tib'


In [ ]:
class MessagePaneHandler(logging.Handler):
    """Buffer decoded messages on the MQTT thread; render on the kernel loop."""
    def __init__(self):
        super().__init__()
        self.lines = deque(maxlen=500)
        self.dirty = False

    def emit(self, record):
        # Handler.handle() holds this handler's lock while appending.
        self.lines.append(self.format(record))
        self.dirty = True


async def refresh_messages(handler, output):
    while True:
        await asyncio.sleep(.5)
        handler.acquire()
        try:
            if not handler.dirty:
                continue
            text = '\n'.join(handler.lines)
            handler.dirty = False
        finally:
            handler.release()
        output.value = '<pre style="margin:0;white-space:pre-wrap">'+escape(text)+'</pre>'


async def stop_monitoring():
    """Dispose of this notebook's displays and connection without board commands."""
    task = globals().get('message_task')
    if task is not None:
        task.cancel()
        with suppress(asyncio.CancelledError):
            await task
    for figure, animation in globals().get('dashboards', {}).values():
        if animation.event_source is not None:
            animation.pause()
        plt.close(figure)
    if 'dashboards' in globals():
        dashboards.clear()
    if 'pcb' in globals():
        await asyncio.to_thread(pcb.close, stop_measurements=False)
    if 'message_handler' in globals():
        message_logger.removeHandler(message_handler)
        message_handler.close()
    if 'message_output' in globals():
        message_output.close()


## Connect and show messages

Rerunning replaces only this notebook's connection and displays. The logger shows
information, warnings, errors, calibration progress, and boot events without
dumping dense throughput packets or duplicating raw debug receive logs.


In [ ]:
await stop_monitoring()
dashboards = {}
message_logger = logging.Logger(f'monitoring.{DEVICE}', level=logging.INFO)
message_logger.propagate = False
message_handler = MessagePaneHandler()
message_handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
message_logger.addHandler(message_handler)
message_output = widgets.HTML(
    value='<i>Waiting for board messages.</i>',
    layout=widgets.Layout(height='280px', overflow='auto', border='1px solid #999'))
pcb = hspcb.HispecFibPcb(BROKER, device=DEVICE, logger=message_logger)
try:
    await asyncio.to_thread(pcb.connect)
    monitor = await asyncio.to_thread(pcb.start_throughput_monitor, 'all', max_samples=40000)
    message_task = asyncio.create_task(refresh_messages(message_handler, message_output))
    display(message_output)
except BaseException:
    await stop_monitoring()
    raise


## YJ dashboard

Run independently of the other channel. Rerunning replaces only this display.


In [ ]:
if 'monitor' not in globals() or not monitor._running.is_set():
    raise RuntimeError('Run the connection cell first.')
if 'yj' in dashboards:
    old_figure, old_animation = dashboards['yj']
    if old_animation.event_source is not None:
        old_animation.pause()
    plt.close(old_figure)
with plt.ioff():
    yj_figure, yj_animation = monitor.plot_live(channel='yj', interval_s=.5, max_points=600)
dashboards['yj'] = (yj_figure, yj_animation)
display(yj_figure.canvas)
yj_figure.canvas.draw()


## HK dashboard

Run independently of the other channel. Rerunning replaces only this display.


In [ ]:
if 'monitor' not in globals() or not monitor._running.is_set():
    raise RuntimeError('Run the connection cell first.')
if 'hk' in dashboards:
    old_figure, old_animation = dashboards['hk']
    if old_animation.event_source is not None:
        old_animation.pause()
    plt.close(old_figure)
with plt.ioff():
    hk_figure, hk_animation = monitor.plot_live(channel='hk', interval_s=.5, max_points=600)
dashboards['hk'] = (hk_figure, hk_animation)
display(hk_figure.canvas)
hk_figure.canvas.draw()


## Optional status query

Leave disabled for a listener that never publishes commands. This queries board
status only; `pd()` is deliberately absent because it enables/rearms PD power.


In [ ]:
QUERY_STATUS = False
if QUERY_STATUS:
    display(await asyncio.to_thread(pcb.status))


## Close this listener

Set the flag when finished. Shutdown closes only these displays, collectors and
MQTT connection. Measurements owned by other notebooks continue.


In [ ]:
STOP_MONITORING = False
if STOP_MONITORING:
    await stop_monitoring()
